# Tratamento dos Dados

## <font color = #FFB90F>**Importações**

### <font color = #56A5EC>**Instalações**

In [1]:
%pip install scikit-survival --quiet
%pip install seaborn --quiet
%pip install --upgrade lifelines --quiet

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


### <font color = #56A5EC>**Importações**

In [2]:
# === Visualização ===
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

# === Manipulação de Dados ===
import numpy as np
import pandas as pd

# === Scikit-learn ===
# Pré-Processamento
from sklearn import set_config
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score)
from sksurv.metrics import integrated_brier_score
from sksurv.util import Surv
set_config(display = "text")

# === Lifelines ===
# Modelagem e Estimações
from lifelines import CoxPHFitter, KaplanMeierFitter,       NelsonAalenFitter
from lifelines.statistics import proportional_hazard_test, multivariate_logrank_test
from lifelines.calibration import survival_probability_calibration

## <font color = #FFB90F>**Tratamento dos Dados**

### <font color = #56A5EC>**Preparação dos Dados**

In [3]:
def data_preparing(df):
    """
    Descrição:
      Prepara o conjunto de dados para análise, aplicando filtros e transformações.

    Parâmetros:
      df: pandas.DataFrame
        O conjunto de dados que será preparado.

    Retorno:
      pandas.DataFrame
        O conjunto de dados após as transformações.
    """

    # Cópia do DataFrame original
    df_aux = df.copy()

    # Redefinição dos índices
    df_aux = df_aux.reset_index(drop = True)

    # ============================ #
    # === SELEÇÕES NAS COLUNAS === #
    # ============================ #

    # Seleção 1 - Topografia Colorretal (C18, C19, C20)
    df_aux = df_aux[df_aux.TOPOGRUP.isin(['C18', 'C19', 'C20'])]

    # Seleção 2 - Residentes de SP
    df_aux = df_aux[df_aux.UFRESID == 'SP']

    # Seleção 3 - Casos com confirmação microscópica
    df_aux = df_aux[df_aux.BASEDIAG == 3]

    # Seleção 4 - ECGRUP's = I, II, III, IV
    df_aux = df_aux[df_aux.ECGRUP.isin(['I', 'II', 'III', 'IV'])]

    # Seleção 5 - ANODIAG até 2019
    df_aux = df_aux[df_aux.ANODIAG <= 2019]

    # Seleção 6 - IDADE maior que 19
    df_aux = df_aux[df_aux.IDADE > 19]

    # Seleção 7 – CATEATEND com grupos SUS e SEM INFORMAÇÃO
    df_aux = df_aux[df_aux.CATEATEND.isin([2, 9])]

    # ========================== #
    # === AJUSTE DAS COLUNAS === #
    # ========================== #

    # Ajuste 1 - Número da DRS
    drs_num = df_aux["DRS"].astype(str).str.extract(r"(\d+)")[0]
    df_aux["nDRS"] = pd.to_numeric(drs_num, errors = "coerce")
    df_aux = df_aux[df_aux["nDRS"].notna()]
    df_aux["nDRS"] = df_aux["nDRS"].astype(int)


    # Ajuste 2 - Recalcular Variáveis
    ## Ajuste 2.1 - Colunas de data para o formato datetime
    list_datas = ['DTCONSULT', 'DTDIAG', 'DTTRAT', 'DTULTINFO']
    for col_data in list_datas:
        df_aux[col_data] = pd.to_datetime(df_aux[col_data], errors = "coerce")

    ## Ajuste 2.2 - Calcular a diferença entre as datas para criar novas variáveis
    df_aux['CONSDIAG'] = (df_aux.DTDIAG    - df_aux.DTCONSULT).dt.days
    df_aux['TRATCONS'] = (df_aux.DTTRAT    - df_aux.DTCONSULT).dt.days
    df_aux['DIAGTRAT'] = (df_aux.DTTRAT    - df_aux.DTDIAG).dt.days
    df_aux['ULTIDIAG'] = (df_aux.DTULTINFO - df_aux.DTDIAG).dt.days

    ## Ajuste 2.3 - Preencher os valores NaN com -1
    df_aux.fillna({'CONSDIAG': -1, 'TRATCONS': -1, 'DIAGTRAT': -1}, inplace = True)

    ## Ajuste 2.4 - Implementar a categorização das variáveis calculadas
    df_aux['CONSDIAG_CAT'] = [3 if consdiag < 0 else 0 if consdiag <= 30 else 1 if consdiag <= 60 else 2 for consdiag in df_aux.CONSDIAG]
    df_aux['TRATCONS_CAT'] = [3 if tratcons < 0 else 0 if tratcons <= 60 else 1 if tratcons <= 90 else 2 for tratcons in df_aux.TRATCONS]
    df_aux['DIAGTRAT_CAT'] = [3 if diagtrat < 0 else 0 if diagtrat <= 60 else 1 if diagtrat <= 90 else 2 for diagtrat in df_aux.DIAGTRAT]


    # Ajuste 3 - Assistência de Alta Complexidade em Oncologia
    ## 3.1 - Agrupamento das colunas de HABILIT
    condlist = [
        df_aux['HABILIT'].isin([1, 3, 5, 12]),
        df_aux['HABILIT'].isin([6, 7]),
        df_aux['HABILIT'].isin([2, 9, 10, 15])
    ]

    ## 3.2 - Nome dos grupos
    choicelist = [
        0,  # UNACON SEM RADIOTERAPIA
        1,  # CACON
        2   # UNACON COM RADIOTERAPIA
    ]

    ## 3.3 - Tudo aquilo que não está na seleção
    default = -1

    ## 3.4 - Seleção das colunas de HABILIT e ajuste do nome das colunas
    df_aux['HABILIT'] = np.select(condlist, choicelist, default).astype(int)

    ## 3.5 - Excluir tudo aquilo que não é das seleções do Ajuste 3
    df_aux = df_aux[df_aux['HABILIT'] != -1]

    # ========================== #
    # === CRIAÇÃO DE COLUNAS === #
    # ========================== #

    # Criação 1 - Coluna para Morfologia do Câncer Colorretal
    ## 1.1 - Atribuição dos valores aos códigos
    COD_ADENO, COD_SINETE, COD_MUCINOSO, COD_INDIFEREN = 81403, 84903, 84803, 80203

    ## 1.2 - Seleção dos códigos
    codigos = [COD_ADENO, COD_SINETE, COD_MUCINOSO, COD_INDIFEREN]
    df_aux = df_aux[df_aux['MORFO'].isin(codigos)]

    ## 1.3 - Criação dos grupos/nomes da coluna de Morfologia
    morfo_map = {
        COD_ADENO:     'Adenocarcinoma',  # 8140/3
        COD_SINETE:    'Anel de sinete',  # 8490/3
        COD_MUCINOSO:  'Mucinoso',        # 8480/3
        COD_INDIFEREN: 'Indiferenciado'   # 8020/3
    }

    df_aux['MORFO_CAT'] = df_aux['MORFO'].map(morfo_map)

    # ================================= #
    # === COLUNAS A SEREM REMOVIDAS === #
    # ================================= #

    drop_cols = [
        'UFNASC','UFRESID','CIDADE','DTCONSULT','CLINICA','DTDIAG','BASEDIAG','DESCTOPO','MORFO','DESCMORFO',
        'T','N','M','PT','PN','PM','S','G','LOCALTNM','IDMITOTIC','PSA','GLEASON','OUTRACLA',
        'META01','META02','META03','META04','DTTRAT','NAOTRAT','TRATAMENTO','TRATHOSP','TRATFANTES','TRATFAPOS',
        'NENHUMANT','CIRURANT','RADIOANT','QUIMIOANT','HORMOANT','TMOANT','IMUNOANT','OUTROANT',
        'NENHUMAPOS','CIRURAPOS','RADIOAPOS','QUIMIOAPOS','HORMOAPOS','TMOAPOS','IMUNOAPOS','OUTROAPOS',
        'DTULTINFO','CICI','CICIGRUP','CICISUBGRU','LATERALI','INSTORIG','PERDASEG','ERRO','DTRECIDIVA',
        'RECNENHUM','RECLOCAL','RECREGIO','RECDIST','REC01','REC02','REC03','REC04','DSCINST','CIDO','DESCIDO',
        'HABILIT2','HABIT11','HABILIT1','CIDADEH','DRS','DRS_INST','CIDADE_INS',
        'NENHUM','CIRURGIA','RADIO','QUIMIO','HORMONIO','IMUNO','OUTROS','RRAS',
        'TMO','TOPOGRUP','EC','IBGE','IBGEATEN','RRAS_INST','FAIXAETAR','CONSDIAG','DIAGTRAT','TRATCONS'
    ]


    # Seleção final das colunas do novo DataFrame
    col = df_aux.columns.drop(drop_cols)

    return df_aux[col]

In [4]:
# Atribuição do Banco de Dados Original
df = pd.read_csv('../DataSet/pacigeral_2025.csv', low_memory = False)

# Exibição
display(df.head(3))
print(df.shape)

,INSTITU,ESCOLARI,IDADE,SEXO,UFNASC,UFRESID,IBGE,CIDADE,CATEATEND,DTCONSULT,...,DRS_INST,CIDADE_INS,CIDO,DESCIDO,RRAS_INST,HABILIT,HABIT11,HABILIT1,HABILIT2,CIDADEH
0,48593,1,0,1,SP,GO,5200050,ABADIA DE GOIAS,9,2008-11-17,...,DRS 07 Campinas,CAMPINAS,89003.0,"RABDOMIOSSARCOMA, SOE",15,15,UNACON exclusiva de Oncologia Pediátrica com S...,2,1,Campinas
1,9,1,0,1,MT,MT,5100102,ACORIZAL,9,2003-03-07,...,DRS 01 Grande Sao Paulo,SAO PAULO,95103.0,"RETINOBLASTOMA, SOE",6,15,UNACON exclusiva de Oncologia Pediátrica com S...,2,1,São Paulo
2,8672,1,0,1,AC,AC,1200013,ACRELANDIA,9,2006-06-15,...,DRS 01 Grande Sao Paulo,SAO PAULO,98613.0,"LEUCEMIA MIELOIDE AGUDA, SOE",6,7,CACON com Serviço de Oncologia Pediátrica,3,2,São Paulo


(1317509, 105)


In [5]:
# Banco de Dados Preparado
df_colo = data_preparing(df)

# Exibição
display(df_colo.head(3))
print(df_colo.shape)

,INSTITU,ESCOLARI,IDADE,SEXO,CATEATEND,DIAGPREV,TOPO,ECGRUP,ULTINFO,ANODIAG,HABILIT,nDRS,ULTIDIAG,CONSDIAG_CAT,TRATCONS_CAT,DIAGTRAT_CAT,MORFO_CAT
33774,9326,2,20,1,2,1,C182,IV,3,2019,1,9,966,2,2,0,Adenocarcinoma
33956,12,4,20,2,9,2,C209,IV,3,2008,2,7,95,3,0,0,Adenocarcinoma
34103,16624,5,20,2,2,2,C189,IV,3,2014,1,1,759,3,2,2,Adenocarcinoma


(32664, 17)


In [6]:
df_colo.columns.tolist()

['INSTITU',
 'ESCOLARI',
 'IDADE',
 'SEXO',
 'CATEATEND',
 'DIAGPREV',
 'TOPO',
 'ECGRUP',
 'ULTINFO',
 'ANODIAG',
 'HABILIT',
 'nDRS',
 'ULTIDIAG',
 'CONSDIAG_CAT',
 'TRATCONS_CAT',
 'DIAGTRAT_CAT',
 'MORFO_CAT']

### <font color = #56A5EC>**Preparação das Preditoras**

In [7]:
def pred_cols(df_colo):
    """
    Descrição:
      Formatação das colunas que são usadas para a predição.

    Parâmetros:
      df_colo: pandas.DataFrame
        O conjunto de dados já preparado.

    Retorno:
      pandas.DataFrame
        O conjunto de dados com as colunas de predição formatadas.
    """

    # Cópia do DataFrame
    df_aux = df_colo.copy()

    # Redefinição dos índices
    df_aux = df_aux.reset_index(drop = True)

    # ===================== #
    # === ULTIDIAG/TIME === #
    # ===================== #

    # Eliminação das linhas onde 'ULTIDIAG' é menor que zero
    df_aux = df_aux[df_aux.ULTIDIAG >=  0]

    # Dividir a coluna de tempo ('ULTIDIAG') em meses
    df_aux['time'] = np.ceil(df_aux['ULTIDIAG'] / 30).astype(int)

    # Ajustar valores menores ou iguais a zero para 1
    df_aux.loc[df_aux['time'] == 0, 'time'] = 1

    # Eliminação da coluna 'ULTIDIAG' (já transformada em 'time')
    df_aux = df_aux.drop('ULTIDIAG', axis = 1)

    # ===================== #
    # === ULTINFO/EVENT === #
    # ===================== #

    # Ajuste da coluna 'ULTINFO' para ser binária (1 = morte / 0 = vivo)
    df_aux['event'] = df_aux['ULTINFO'].apply(lambda x: 1 if x in [3, 4] else 0)

    # Ajustar valores maiores que 60 meses (5 anos) para 61 e definir evento como 0 (censurando os casos que fogem ao escopo)
    df_aux.loc[df_aux['time'] > 60, ['time', 'event']] = [61, 0]

    # Eliminar a coluna 'ULTINFO' (agora transformada para 'event')
    df_aux = df_aux.drop('ULTINFO', axis = 1)

    return df_aux

In [8]:
# Aplicação da preparação das colunas preditivas
df_colo = pred_cols(df_colo)

In [9]:
# Visualização das colunas com o evento
df_colo['event'].value_counts().sort_index()

event
0    14524
1    18140
Name: count, dtype: int64

In [10]:
# Visualização da coluna de evento
df_colo['time'].value_counts().sort_index()

time
1      1188
2      1007
3       884
4       812
5       714
      ...  
57      141
58      141
59      140
60      158
61    12412
Name: count, Length: 61, dtype: int64

In [11]:
# Total de Valores Censurados
censored_count = df_colo['event'].value_counts()[0]
print(f"Número de valores censurados: {censored_count}")

Número de valores censurados: 14524


## <font color = #FFB90F>**Download**

In [13]:
df_colo.to_csv('Colorretal_tratado_2025.csv', index=False)